# Introduction to Qiskit

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import RZGate, HGate
import numpy as np

# QuantumCircuits

In Qiskit, quantum circuits are created using the ```QuantumCircuit``` object. When instantiating a ```QuantumCircuit```, you must specify the number of qubits of your circuit.

You can visualize a circuit at any time using the ```draw()``` method. By default, circuits are drawn in ASCII, however passing ```'mpl'``` as an argument produces drawings using matplotlib, which are generally easier to interpret.


In [ ]:
num_qubits = 3

# Create a quantum circuit with a specified number of qubits
qc = QuantumCircuit(num_qubits)

# Display the quantum circuit
qc.draw("mpl")

Additionally, you may instantiate a ```QuantumCircuit``` by passing it one or multiple ```QuamtumRegister``` objects, and optionally some ```ClassicalRegister```objects. The classical register serves to store measurement information.

In [ ]:
# Instantiate multiple quantum registers
qreg_1 = QuantumRegister(size=3, name="A") 
qreg_2 = QuantumRegister(size=2, name="B")
# Instantiate a classical register
creg = ClassicalRegister(size=5)

# Use registers to initialize the quantum circuit
qc = QuantumCircuit(qreg_1, qreg_2, creg)

# Display the quantum circuit
qc.draw("mpl")

## Quantum gates

### Single qubit gates

Any operation which can be performed on a quantum circuit in Qiskit is represented by an ```Instruction```, and quantum unitary gates are a subclass of ```Instruction``` called ```Gate```. For most common gates, ```QuantumCircuit``` has a method which applies the gate directly to the qubit(s) passed as argument. For rotation gates, a rotation angle must also be specified.  Naturally, the order in which gates are appended to the circuit determines what order they will be applied to the quantum state.

It may be practical to instantiate a ```Gate``` directly, in which case it can be added to the circuit using the ```append()``` method.

In [ ]:
# Add gates to the circuit
# X gate on the zeroth qubit
qc.x(0)
# Y rotation of PI on qubit one
qc.ry(np.pi, 1)

# Instantiate a Gate object and add it to qubit 2. The qubit(s) indices argument must be an iterable
rz = RZGate(np.pi / 2)
qc.append(rz, [2])
# Display the quantum circuit
qc.draw("mpl")

### Controlled Gates
Just like single-qubit gates, common multi-qubit gates have their associated methods. In this case, it is also necessary to pass the indices of the control qubit(s) as an argument. 

Generally, any $m$-qubit ```Gate``` can be made into its controlled equivalent using the ```control()``` method (it can be made to be controlled by any $n$ number of qubits and by any basis state in the computational basis). When appending a controlled gate to a quantum circuit, a list of qubits must be specified, where the first $n$ are the control qubits, and the last $m$ are acted upon.

In [ ]:
# Add controlled gates to the circuit
# CNOT controlled by qubit 0 on qubit 1
qc.cx(0, 1)
# Controlled y rotation by pi/2 on qubit 0, controlled by qubit 3
qc.cry(np.pi / 2, 2, 0)

# Create a multicontrolled hadamard gate, controlled by the |10> state on qubits 0 and 1, and applied on qubit 2
mch = HGate().control(2, ctrl_state="10")
qc.append(mch, range(3))
qc.draw("mpl")

### Exercices

Using the tools presented so far, please build and draw the following circuits:

#### 1.

 ![circuit1](./images/circuit_1.png)

 Note that the two two-qubit gates in this circuit are a controlled $Z$ gate and a SWAP, respectively

In [ ]:
###Your code here###

#### 2. 

 ![circuit1](./images/circuit_2.png)

In [ ]:
###Your code here###

#### 3. 

 ![circuit1](./images/circuit_3.png)

In [ ]:
###Your code here###

# Parameterized quantum circuits

In the context of variational algorithms, it is often necessary to execute multiple experiments with the same circuit architecture, where the gate parameters are updated between runs. Rather than having to rebuild the circuit every time, we are able to create parameterized circuits where the values of the gate parameters need not be specified until execution. 

Parameterized gates and circuits can be instantiated using the ```Parameter``` object.

In [ ]:
from qiskit.circuit import Parameter, ParameterVector

# Create a pair of parameter objects
theta = Parameter("θ")
phi = Parameter("φ")

param_qc = QuantumCircuit(3)

# Add a parameterized ry gate
param_qc.ry(theta, 0)

# Add a parameterized controlled Z rotation
p_cry = RZGate(phi).control(1)
param_qc.append(p_cry, [2, 1])

param_qc.draw("mpl")

When a large number of parameters is needed, it is more practical to use a ```ParameterVector```

In [ ]:
# Instantiate a ParameterVector of length 6
beta = ParameterVector("β", 6)

# Append many parameterized gates to the circuit
for i in range(3):
    param_qc.ry(beta[2 * i], i)
    param_qc.rz(beta[2 * i + 1], i)
param_qc.draw("mpl")

### Assigning Parameters

Once a parameterized circuit has been created, its parameters need to be assigned numerical values before it can be run on quantum hardware. One way to do this is through the ```assign_parameters()``` method. By default, this method is not in place, and returns another instance of ```QuantumCircuit```.

There are two possible inputs to ```assign_parameters()```. If you give it a list of values, the values will be bound to the parameters in the order in which they are stored in the circuit object. This is in alphabetical order by name, and NOT by the order of the gates in the circuit! Be mindful of which values are assigned to which parameters!

If you wish to avoid misordering problems, you may instead pass it a dictionary of the form ```{Parameter: value, ParametersVector: [values]}```.

In [ ]:
# check the order in which parameters are stored
print(param_qc.parameters)

# assign parameters using list
assigned_qc_list = param_qc.assign_parameters([1, 2, 3, 4, 5, 6, 7, 8])

# assign parameters using dictionary
param_qc.assign_parameters({beta: [1, 2, 3, 4, 5, 6], theta: 7, phi: 8}, inplace=True)

# verify that both methods are equivalent
print(assigned_qc_list == param_qc)

assigned_qc_list.draw("mpl")

### Exercise

Prepare the following parameterized ansatz:

  ![ansatz](./images/ansatz.png)

 then find and assign it the parameter values which will yield the following statevector

  ![circuit1](./images/statevec.png)

  You can use the ```plot_statevector()``` method provided to visualize the statevector produced by your circuit. 


In [ ]:
from utils import plot_statevector

qc = QuantumCircuit(4)
###Your code here###

plot_statevector(qc)